# Technique 6: Standardization / Scaling**Group ID:** 2026-Y2-S1-MET-18**Member:** Jayamanna N.G.K.D | IT25102121Credit Risk Dataset — individual contribution notebook, extracted from `group_pipeline.ipynb`.

In [ ]:
# ==============================# SETUP: Load the Credit Risk dataset# (Shared setup step used across the group pipeline)# ==============================import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsimport kagglehubimport os# Download latest version of the datasetpath = kagglehub.dataset_download("laotse/credit-risk-dataset")print("Path to dataset files:", path)print("\nFiles in dataset folder:")print(os.listdir(path))# Load the datasetfile_path = os.path.join(path, "credit_risk_dataset.csv")df = pd.read_csv(file_path)print("\nDataset loaded successfully!")print(df.head())

In [ ]:
# ==============================# PREREQUISITE: Basic cleaning inherited from the group pipeline# (duplicate removal + missing value handling, done earlier by#  Members 1 and 2, reproduced here so this notebook runs standalone)# ==============================df = df.drop_duplicates()if "person_emp_length" in df.columns:    df["person_emp_length"] = df["person_emp_length"].fillna(        df["person_emp_length"].median()    )if "loan_int_rate" in df.columns:    df["loan_int_rate"] = df["loan_int_rate"].fillna(        df["loan_int_rate"].median()    )print("Shape after cleaning:", df.shape)

In [ ]:
# ==============================# PREREQUISITE: Define X and y (inherited from the group pipeline)# ==============================X = df.drop("loan_status", axis=1)y = df["loan_status"]print("X shape:", X.shape)print("y shape:", y.shape)

---
## Technique 6: Normalization / Scaling
**Section owner:
Jayamanna N.G.K.D | IT25102121**

The code uses StandardScaler to standardize numerical features to a similar scale. This is needed to prevent features with larger values from dominating the model.

In [ ]:
# ==============================
# MEMBER 6
# Train/Test Split + Scaling + Final Preprocessing
# ==============================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer


# --------------------------------
# 1. Train-Test Split
# --------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


# --------------------------------
# 2. Identify columns
# --------------------------------

numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("\nNumerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)


# --------------------------------
# 3. Numerical preprocessing
# --------------------------------

numerical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)


# --------------------------------
# 4. Categorical preprocessing
# --------------------------------

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


# --------------------------------
# 5. Combine preprocessing
# --------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_transformer,
            numerical_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)


# --------------------------------
# 6. Transform training/test data
# --------------------------------

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)


# --------------------------------
# 7. Check final shapes
# --------------------------------

print("\nFinal Processed Data:")

print(
    "X_train_processed shape:",
    X_train_processed.shape
)

print(
    "X_test_processed shape:",
    X_test_processed.shape
)

print(
    "y_train shape:",
    y_train.shape
)

print(
    "y_test shape:",
    y_test.shape
)

X_train shape: (25932, 11)
X_test shape: (6484, 11)
y_train shape: (25932,)
y_test shape: (6484,)

Numerical Features:
['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length']

Categorical Features:
['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']

Final Processed Data:
X_train_processed shape: (25932, 26)
X_test_processed shape: (6484, 26)
y_train shape: (25932,)
y_test shape: (6484,)
